# 환경 설정 및 라이브러리 로드

In [ ]:
import pandas as pd
import numpy as np
import os
import re
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
import ipywidgets as widgets

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

BASE_PATH = r'C:\Users\T\Desktop\SeoulMate'
print('환경 설정 완료')


# 2025년 센서 데이터 전처리

In [ ]:
sensor_all = pd.read_csv(
    rf'{BASE_PATH}\sensor_preprocessed.csv',
    encoding='utf-8-sig'
)

sensor_all = sensor_all.drop(columns=['uv_risk', 'vibration_risk'], errors='ignore')

sensor_all['측정시간'] = pd.to_datetime(sensor_all['측정시간'])
sensor_all['year']  = sensor_all['측정시간'].dt.year
sensor_all['month'] = sensor_all['측정시간'].dt.month

# Heat Index 계산
def calc_heat_index(row):
    T  = row['온도 평균(℃)']
    RH = row['습도 평균(%)']
    if T < 27:
        return T
    return (-8.78469475556 +
            1.61139411 * T +
            2.33854883889 * RH +
            -0.14611605 * T * RH +
            -0.012308094 * T**2 +
            -0.0164248277778 * RH**2 +
            0.002211732 * T**2 * RH +
            0.00072546 * T * RH**2 +
            -0.000003582 * T**2 * RH**2)

sensor_all['heat_index'] = sensor_all.apply(calc_heat_index, axis=1)

# 계절별 온도 이상값 제거
def remove_temp_outlier(row):
    T = row['온도 평균(℃)']
    m = row['month']
    if m in [12, 1, 2]:  return T <= 30
    elif m in [3, 4, 5]: return T <= 35
    elif m in [6, 7, 8]: return T <= 40
    else:                 return T <= 35

mask = sensor_all.apply(remove_temp_outlier, axis=1)
sensor_all = sensor_all[mask]

# 위험 피처 생성
sensor_all['high_temp_risk'] = (sensor_all['heat_index'] > 35).astype(int)
sensor_all['low_temp_risk']  = (sensor_all['온도 평균(℃)'] < 5).astype(int)
sensor_all['humidity_risk']  = (
    (sensor_all['습도 평균(%)'] < 30) | (sensor_all['습도 평균(%)'] > 80)
).astype(int)
sensor_all['noise_risk'] = (sensor_all['소음 평균(dB)'] > 55).astype(int)

# 행정동 이름 통일
dong_mapping = {
    'Deungchon-dong2':    'Deungchon2(i)-dong',
    'Eungam3-dong':       'Eungam3(sam)-dong',
    'Geumho-dong2.3-ga':  'Geumho-2.3(i)(sam)-dong',
    'Hongje2-dong':       'Hongje2(i)-dong',
    'Jamsil3-dong':       'Jamsil3(sam)-dong',
    'Seocho4-dong':       'Seocho4(sa)-dong',
    'Seongnae2-dong':     'Seongnae2(i)-dong',
    'Seongsu1ga2-dong':   'Seongsu1ga2(i)-dong',
    'Siheung3-dong':      'Siheung3(sam)-dong',
}
sensor_all['행정동'] = sensor_all['행정동'].replace(dong_mapping)

print(f'전처리 완료: {len(sensor_all):,}행')
print(f'행정동 고유값: {sensor_all["행정동"].nunique()}개')


# 2025년 에어코리아 데이터 병합

In [ ]:
folder = rf'{BASE_PATH}\2025airkorea'
all_air = []
for file in os.listdir(folder):
    df = pd.read_excel(os.path.join(folder, file), header=0)
    all_air.append(df)
air_df = pd.concat(all_air, ignore_index=True)

seoul_gu = [
    '중구', '종로구', '용산구', '광진구', '성동구', '중랑구', '동대문구',
    '성북구', '도봉구', '은평구', '서대문구', '마포구', '강서구', '구로구',
    '영등포구', '동작구', '관악구', '강남구', '서초구', '송파구', '강동구',
    '금천구', '강북구', '양천구', '노원구'
]
air_seoul = air_df[air_df['측정소명'].isin(seoul_gu)].copy()
air_seoul['측정일시'] = air_seoul['측정일시'].astype(str)
air_seoul['month']   = air_seoul['측정일시'].str[4:6].astype(int)
air_seoul['PM25']    = pd.to_numeric(air_seoul['PM25'], errors='coerce')
air_seoul['PM10']    = pd.to_numeric(air_seoul['PM10'], errors='coerce')

air_monthly = air_seoul.groupby(['측정소명', 'month']).agg(
    pm25_월평균=('PM25', 'mean'),
    pm10_월평균=('PM10', 'mean')
).reset_index()
air_monthly.columns = ['자치구명', 'month', 'pm25_월평균', 'pm10_월평균']

# 결측치 보간
air_monthly[['pm25_월평균', 'pm10_월평균']] = (
    air_monthly.groupby('자치구명')[['pm25_월평균', 'pm10_월평균']]
    .transform(lambda x: x.interpolate())
)

gu_mapping = {
    'Jongno-gu': '종로구', 'Jung-gu': '중구', 'Yongsan-gu': '용산구',
    'Seongdong-gu': '성동구', 'Gwangjin-gu': '광진구', 'Dongdaemun-gu': '동대문구',
    'Jungnang-gu': '중랑구', 'Seongbuk-gu': '성북구', 'Gangbuk-gu': '강북구',
    'Dobong-gu': '도봉구', 'Nowon-gu': '노원구', 'Eunpyeong-gu': '은평구',
    'Seodaemun-gu': '서대문구', 'Mapo-gu': '마포구', 'Yangcheon-gu': '양천구',
    'Gangseo-gu': '강서구', 'Guro-gu': '구로구', 'Geumcheon-gu': '금천구',
    'Yeongdeungpo-gu': '영등포구', 'Dongjak-gu': '동작구', 'Gwanak-gu': '관악구',
    'Seocho-gu': '서초구', 'Gangnam-gu': '강남구', 'Songpa-gu': '송파구',
    'Gangdong-gu': '강동구'
}
sensor_all['자치구명'] = sensor_all['자치구'].map(gu_mapping)
sensor_all = sensor_all.merge(air_monthly, on=['자치구명', 'month'], how='left')

print(f'✅ 병합 완료: {len(sensor_all):,}행')
print(f'pm25_월평균 결측치: {sensor_all["pm25_월평균"].isnull().sum():,}')

# 2025년 위험노출점수 계산 및 동별 월별 집계

In [ ]:
def calc_risk_exposure(row):
    score = 0.0
    if row['pm25_월평균'] > 35:    score += 3 * 0.20
    elif row['pm25_월평균'] > 25:  score += 2 * 0.20
    elif row['pm25_월평균'] > 15:  score += 1 * 0.20

    if row['pm10_월평균'] > 100:   score += 2 * 0.10
    elif row['pm10_월평균'] > 50:  score += 1 * 0.10

    if row['heat_index'] > 40:     score += 3 * 0.18
    elif row['heat_index'] > 35:   score += 2 * 0.18
    elif row['heat_index'] > 30:   score += 1 * 0.18

    if row['온도 평균(℃)'] < 0:    score += 3 * 0.08
    elif row['온도 평균(℃)'] < 5:  score += 2 * 0.08
    elif row['온도 평균(℃)'] < 10: score += 1 * 0.08

    h = row['습도 평균(%)']
    if h < 20 or h > 90:           score += 3 * 0.08
    elif h < 30 or h > 80:         score += 2 * 0.08
    elif h < 40 or h > 70:         score += 1 * 0.08

    if row['소음 평균(dB)'] > 70:   score += 3 * 0.09
    elif row['소음 평균(dB)'] > 65: score += 2 * 0.09
    elif row['소음 평균(dB)'] > 55: score += 1 * 0.09
    return score

sensor_all['위험노출점수'] = sensor_all.apply(calc_risk_exposure, axis=1)

# 동별 월별 집계
monthly_risk_2025 = sensor_all.groupby(['자치구', '행정동', 'year', 'month']).agg(
    위험노출시간=('위험노출점수', 'mean'),
    pm25_평균=('pm25_월평균', 'mean'),
    pm10_평균=('pm10_월평균', 'mean'),
    온도_평균=('온도 평균(℃)', 'mean'),
    습도_평균=('습도 평균(%)', 'mean'),
    열지수_평균=('heat_index', 'mean'),
    소음_평균=('소음 평균(dB)', 'mean'),
    high_temp_risk_비율=('high_temp_risk', 'mean'),
    low_temp_risk_비율=('low_temp_risk', 'mean'),
    humidity_risk_비율=('humidity_risk', 'mean'),
    noise_risk_비율=('noise_risk', 'mean'),
    병원수=('병원수', 'first'),
    공원면적=('공원면적', 'first'),
    금연구역수=('금연구역수', 'first'),
).reset_index()

print(f' 집계 완료: {len(monthly_risk_2025):,}행')
print(monthly_risk_2025['위험노출시간'].describe())

# 보조 피처 병합 (disease, population)

In [ ]:
# disease 로드 (header=3 필수!)
disease = pd.read_excel(rf'{BASE_PATH}\disease.xlsx', header=3)
disease_gu = disease[['시군구', '환자수.4']].copy()
disease_gu.columns = ['자치구명', '환자수']
disease_gu = disease_gu[disease_gu['자치구명'] != '계'].reset_index(drop=True)

# population 로드
population = pd.read_excel(rf'{BASE_PATH}\population.xlsx')
pop_gu = population[
    (population['동별(2)'] == '소계') &
    (population['항목'] == '계') &
    (population['동별(1)'] != '합계')
][['동별(1)', '2025 4/4']].copy()
pop_gu.columns = ['자치구명', '인구수']

disease_gu = disease_gu.merge(pop_gu, on='자치구명', how='left')
disease_gu['환자수_인구보정'] = disease_gu['환자수'] / disease_gu['인구수'] * 100000

# 자치구명 한글 변환 후 병합
gu_mapping_kor = {
    'Jongno-gu': '종로구', 'Jung-gu': '중구', 'Yongsan-gu': '용산구',
    'Seongdong-gu': '성동구', 'Gwangjin-gu': '광진구', 'Dongdaemun-gu': '동대문구',
    'Jungnang-gu': '중랑구', 'Seongbuk-gu': '성북구', 'Gangbuk-gu': '강북구',
    'Dobong-gu': '도봉구', 'Nowon-gu': '노원구', 'Eunpyeong-gu': '은평구',
    'Seodaemun-gu': '서대문구', 'Mapo-gu': '마포구', 'Yangcheon-gu': '양천구',
    'Gangseo-gu': '강서구', 'Guro-gu': '구로구', 'Geumcheon-gu': '금천구',
    'Yeongdeungpo-gu': '영등포구', 'Dongjak-gu': '동작구', 'Gwanak-gu': '관악구',
    'Seocho-gu': '서초구', 'Gangnam-gu': '강남구', 'Songpa-gu': '송파구',
    'Gangdong-gu': '강동구'
}
monthly_risk_2025['자치구명'] = monthly_risk_2025['자치구'].map(gu_mapping_kor)
monthly_risk_2025 = monthly_risk_2025.merge(
    disease_gu[['자치구명', '환자수_인구보정']], on='자치구명', how='left'
)

print(f'보조 피처 병합 완료: {len(monthly_risk_2025):,}행')

# 행정동 이름 → 행정동 코드 매핑

In [ ]:
dong_eng  = pd.read_csv(rf'{BASE_PATH}\dong_eng.csv',  encoding='cp949')
dong_code = pd.read_csv(rf'{BASE_PATH}\dong_code.csv', encoding='cp949')

monthly_risk = monthly_risk_2025.copy()
eng_to_kor   = dict(zip(dong_eng['읍면동영문명'], dong_eng['읍면동명']))

def add_space_before_number(name):
    return re.sub(r'(\D)(\d)', r'\1 \2', name)

def fix_eng_name(name):
    name = name.replace('Gu-ro',     'Guro')
    name = name.replace('Sang-do',   'Sangdo')
    name = name.replace('Dapsip-ri', 'Dapsipri')
    name = name.replace('Myeon-gil', 'Myeongil')
    return name

monthly_risk['행정동_eng_fixed']  = monthly_risk['행정동'].apply(add_space_before_number)
monthly_risk['행정동_eng_fixed2'] = monthly_risk['행정동_eng_fixed'].apply(fix_eng_name)
monthly_risk['행정동_한글']       = monthly_risk['행정동_eng_fixed2'].map(eng_to_kor)

# 수동 매핑
manual_mapping = {
    'Chang 5-dong': '창5동', 'Dapsipri 1(il)-dong': '답십리1동',
    'Dapsipri 2(i)-dong': '답십리2동', 'Yongshin-dong': '용신동',
    'Sangil-dong': '상일동', 'Ilwon 1(il)-dong': '일원1동',
    'Ilwon 2(i)-dong': '일원2동', 'Ilwonbon-dong': '일원본동',
    'Nakseongdae-dong': '낙성대동', 'Seorim-dong': '서림동',
    'Seowon-dong': '서원동', 'Sinllim-dong': '신림동',
    'CheongwoonHyoja-dong': '청운효자동', 'Jeonnong 3(sam)-dong': '전농3동',
    'Jong-ro 1(il). 2(i). 3(sam': '종로1?2?3?4가동',
    'Jong-ro 5(o). 6(yuk)-ga-d': '종로5?6가동',
    'Eulji-ro-dong': '을지로동', 'Gwanghui-dong': '광희동',
    'Man-gu 3(sam)-dong': '망우3동', 'Myeonmok 3(sam). 8(pal)-d': '면목3?8동',
    'Gongdeok-dong': '공덕동', 'Sogang-dong': '서강동',
    'Sanggye 3(sam). 4(sa)-don': '상계3?4동',
    'Sanggye 6(yuk). 7(chil)-d': '상계6?7동',
    'Banpo 1-dong': '반포1동', 'Banpo 2-dong': '반포2동',
    'Banpo 3-dong': '반포3동', 'Bukahyeon-dong': '북아현동',
    'Gileum 1(il)-dong': '길음1동', 'Gileum 2(i)-dong': '길음2동',
    'Geumho- 1(il)-dong': '금호1가동', 'Geumho- 2. 3(i)(sam)-dong': '금호2?3가동',
    'Geumho-dong 4-ga': '금호4가동', 'Geumho 4(sa)-dong': '금호4가동',
    'Seongsu 1ga 1(il)-dong': '성수1가1동', 'Seongsu 1ga 2(i)-dong': '성수1가2동',
    'Seongsu 2ga 1(il)-dong': '성수2가1동', 'Seongsu 2ga 3(sam)-dong': '성수2가3동',
    'Wangsimni-doSeondong': '왕십리도선동', 'Wirye-dong': '위례동',
    'Dangsan 2(i)-dong': '당산2동', 'Munllae-dong': '문래동',
    'Sin-gil 1(il)-dong': '신길1동', 'Sin-gil 3(sam)-dong': '신길3동',
    'Sin-gil 4-dong': '신길4동', 'Sin-gil 5(o)-dong': '신길5동',
    'Sin-gil 6(yuk)-dong': '신길6동', 'Sin-gil 7(chil)-dong': '신길7동',
    'Yeouido-dong': '여의동', 'Hangang-ro-dong': '한강로동',
    'Wonhyo-ro- 1(il)-dong': '원효로1동', 'Wonhyo-ro- 2(i)-dong': '원효로2동',
    'Gahoe-dong': '가회동',
}
monthly_risk['행정동_한글'] = monthly_risk.apply(
    lambda row: manual_mapping.get(row['행정동_eng_fixed2'], row['행정동_한글']), axis=1
)

# 특수문자 통일 및 용산2동 보정
monthly_risk['행정동_한글'] = monthly_risk['행정동_한글'].str.replace('·', '?')
monthly_risk.loc[monthly_risk['행정동_한글'] == '용산2동', '행정동_한글'] = '용산2가동'

# dong_code 병합 및 전농3동 제거
dong_code_merge = dong_code[['행정동_코드', '행정동_명']].copy()
monthly_risk = monthly_risk.merge(
    dong_code_merge, left_on='행정동_한글', right_on='행정동_명', how='left'
)
monthly_risk = monthly_risk[monthly_risk['행정동_코드'].notnull()].reset_index(drop=True)

# 상일동 → 상일제1동 + 상일제2동 분리
sangil  = monthly_risk[monthly_risk['행정동_한글'] == '상일동'].copy()
sangil1 = sangil.copy(); sangil1['행정동_한글'] = '상일제1동'; sangil1['행정동_코드'] = 1174052500
sangil2 = sangil.copy(); sangil2['행정동_한글'] = '상일제2동'; sangil2['행정동_코드'] = 1174052600
monthly_risk = monthly_risk[monthly_risk['행정동_한글'] != '상일동']
monthly_risk = pd.concat([monthly_risk, sangil1, sangil2], ignore_index=True)

print(f'✅ 코드 매핑 완료: {len(monthly_risk):,}행, {monthly_risk["행정동_코드"].nunique()}개 동')

# 2026년 센서 데이터 전처리 및 병합

In [ ]:
folder_2026 = r'C:\Users\T\Desktop\2025_2026.sensor\2026'
all_2026 = []
for file in os.listdir(folder_2026):
    if file.endswith('.csv'):
        df = pd.read_csv(os.path.join(folder_2026, file), encoding='cp949', index_col=False)
        all_2026.append(df)
sensor_2026_raw = pd.concat(all_2026, ignore_index=True)

# 필요한 컬럼 추출 및 전처리
sensor_2026 = sensor_2026_raw[[
    '측정시간', '자치구', '행정동',
    '온도 평균(℃)', '습도 평균(%)',
    '자외선 평균(UV)', '소음 평균(dB)',
    '진동(x) 평균(mm/s)', '진동(y) 평균(mm/s)', '진동(z) 평균(mm/s)'
]].copy()

sensor_2026['측정시간'] = sensor_2026['측정시간'].str.replace('_', ' ')
sensor_2026['측정시간'] = pd.to_datetime(sensor_2026['측정시간'])
sensor_2026['year']  = sensor_2026['측정시간'].dt.year
sensor_2026['month'] = sensor_2026['측정시간'].dt.month

# 서울 자치구 필터
seoul_gu_eng = list(gu_mapping.keys())
sensor_2026  = sensor_2026[sensor_2026['자치구'].isin(seoul_gu_eng)].reset_index(drop=True)

# heat_index 및 위험 피처 계산
sensor_2026['heat_index']    = sensor_2026.apply(calc_heat_index, axis=1)
mask_2026 = sensor_2026.apply(remove_temp_outlier, axis=1)
sensor_2026 = sensor_2026[mask_2026].reset_index(drop=True)

sensor_2026['high_temp_risk'] = (sensor_2026['heat_index'] > 35).astype(int)
sensor_2026['low_temp_risk']  = (sensor_2026['온도 평균(℃)'] < 5).astype(int)
sensor_2026['humidity_risk']  = (
    (sensor_2026['습도 평균(%)'] < 30) | (sensor_2026['습도 평균(%)'] > 80)
).astype(int)
sensor_2026['noise_risk'] = (sensor_2026['소음 평균(dB)'] > 55).astype(int)
sensor_2026['행정동'] = sensor_2026['행정동'].replace(dong_mapping)

# 2025년 에어코리아 1~4월 평균으로 pm25/pm10 보완
air_monthly_14 = air_seoul[air_seoul['month'].isin([1,2,3,4])].groupby(['측정소명', 'month']).agg(
    pm25_월평균=('PM25', 'mean'),
    pm10_월평균=('PM10', 'mean')
).reset_index()
air_monthly_14.columns = ['자치구명', 'month', 'pm25_월평균', 'pm10_월평균']

sensor_2026['자치구명'] = sensor_2026['자치구'].map(gu_mapping)
sensor_2026 = sensor_2026.merge(air_monthly_14, on=['자치구명', 'month'], how='left')
sensor_2026[['pm25_월평균', 'pm10_월평균']] = (
    sensor_2026.groupby('자치구명')[['pm25_월평균', 'pm10_월평균']]
    .transform(lambda x: x.interpolate())
)

# 동별 월별 집계
common_cols = ['자치구', '행정동', '자치구명', 'year', 'month',
               '위험노출시간', 'pm25_평균', 'pm10_평균', '온도_평균', '습도_평균',
               '열지수_평균', '소음_평균', 'high_temp_risk_비율', 'low_temp_risk_비율',
               'humidity_risk_비율', 'noise_risk_비율']

agg_2026 = sensor_2026.groupby(['자치구', '행정동', '자치구명', 'year', 'month']).agg(
    위험노출시간=('소음 평균(dB)', 'mean'),  # placeholder, 아래에서 재계산
    pm25_평균=('pm25_월평균', 'mean'),
    pm10_평균=('pm10_월평균', 'mean'),
    온도_평균=('온도 평균(℃)', 'mean'),
    습도_평균=('습도 평균(%)', 'mean'),
    열지수_평균=('heat_index', 'mean'),
    소음_평균=('소음 평균(dB)', 'mean'),
    high_temp_risk_비율=('high_temp_risk', 'mean'),
    low_temp_risk_비율=('low_temp_risk', 'mean'),
    humidity_risk_비율=('humidity_risk', 'mean'),
    noise_risk_비율=('noise_risk', 'mean'),
).reset_index()

print(f'2026년 집계 완료: {len(agg_2026):,}행, {agg_2026["행정동"].nunique()}개 동')

# 2025 + 2026 통합 및 건강위험도 점수 산출

In [ ]:
# 2025년 데이터에서 보조 피처 추출
aux_cols = ['행정동', '자치구명', '병원수', '공원면적', '금연구역수',
            '환자수_인구보정', '행정동_한글', '행정동_코드']
aux = monthly_risk[aux_cols].drop_duplicates(subset=['행정동', '자치구명']).reset_index(drop=True)

# 2026 보조 피처 병합
agg_2026 = agg_2026.merge(aux, on=['행정동', '자치구명'], how='left')
agg_2026 = agg_2026[agg_2026['행정동'] != 'Jeonnong3(sam)-dong'].reset_index(drop=True)

# 2025 + 2026 합치기
monthly_risk['계절'] = monthly_risk['month'].apply(
    lambda m: '봄' if m in [3,4,5] else '여름' if m in [6,7,8] else '가을' if m in [9,10,11] else '겨울'
)
agg_2026['계절'] = agg_2026['month'].apply(
    lambda m: '봄' if m in [3,4,5] else '여름' if m in [6,7,8] else '가을' if m in [9,10,11] else '겨울'
)

df_all = pd.concat([monthly_risk, agg_2026], ignore_index=True)
df_all = df_all[df_all['year'] != 2023].reset_index(drop=True)

# 위험노출시간 재계산 (집계 컬럼 기준)
def calc_risk_exposure_agg(row):
    score = 0.0
    if row['pm25_평균'] > 35:    score += 3 * 0.20
    elif row['pm25_평균'] > 25:  score += 2 * 0.20
    elif row['pm25_평균'] > 15:  score += 1 * 0.20
    if row['pm10_평균'] > 100:   score += 2 * 0.10
    elif row['pm10_평균'] > 50:  score += 1 * 0.10
    if row['열지수_평균'] > 40:   score += 3 * 0.18
    elif row['열지수_평균'] > 35: score += 2 * 0.18
    elif row['열지수_평균'] > 30: score += 1 * 0.18
    if row['온도_평균'] < 0:      score += 3 * 0.08
    elif row['온도_평균'] < 5:    score += 2 * 0.08
    elif row['온도_평균'] < 10:   score += 1 * 0.08
    h = row['습도_평균']
    if h < 20 or h > 90:         score += 3 * 0.08
    elif h < 30 or h > 80:       score += 2 * 0.08
    elif h < 40 or h > 70:       score += 1 * 0.08
    if row['소음_평균'] > 70:     score += 3 * 0.09
    elif row['소음_평균'] > 65:   score += 2 * 0.09
    elif row['소음_평균'] > 55:   score += 1 * 0.09
    return score

df_all['위험노출시간'] = df_all.apply(calc_risk_exposure_agg, axis=1)

# 정규화 및 점수 계산
scaler = MinMaxScaler()
df_all['위험노출시간_norm'] = scaler.fit_transform(df_all[['위험노출시간']])
df_all['환자수_norm']       = scaler.fit_transform(df_all[['환자수_인구보정']])
df_all['병원수_norm']       = 1 - scaler.fit_transform(df_all[['병원수']])
df_all['공원면적_norm']     = 1 - scaler.fit_transform(df_all[['공원면적']])
df_all['금연구역수_norm']   = 1 - scaler.fit_transform(df_all[['금연구역수']])

df_all['건강위험도_점수'] = (
    df_all['위험노출시간_norm'] * 0.73 +
    df_all['병원수_norm']       * 0.10 +
    df_all['환자수_norm']       * 0.07 +
    df_all['공원면적_norm']     * 0.08 +
    df_all['금연구역수_norm']   * 0.05 - 0.03
) * 100
df_all['건강위험도_점수'] = df_all['건강위험도_점수'].clip(0, 100)

# 분위수 기반 5단계 등급
q20 = df_all['건강위험도_점수'].quantile(0.20)
q40 = df_all['건강위험도_점수'].quantile(0.40)
q60 = df_all['건강위험도_점수'].quantile(0.60)
q80 = df_all['건강위험도_점수'].quantile(0.80)

def assign_grade(score):
    if score <= q20:   return 1
    elif score <= q40: return 2
    elif score <= q60: return 3
    elif score <= q80: return 4
    else:              return 5

df_all['건강위험도_등급'] = df_all['건강위험도_점수'].apply(assign_grade)

print(f'점수 계산 완료!')
print(df_all['건강위험도_점수'].describe())
print(f'\n등급 분포:')
print(df_all['건강위험도_등급'].value_counts().sort_index())
print(f'\n연도별 평균:')
print(df_all.groupby('year')['건강위험도_점수'].mean().round(1))

# 누락 행정동 보완 및 최종 저장

In [ ]:
# 필요한 컬럼만 정리
final_cols = [
    '행정동_코드', '행정동_한글', '자치구명', 'year', 'month',
    '위험노출시간', 'pm25_평균', 'pm10_평균', '온도_평균', '습도_평균',
    '열지수_평균', '소음_평균', 'high_temp_risk_비율', 'low_temp_risk_비율',
    'humidity_risk_비율', 'noise_risk_비율', '병원수', '공원면적', '금연구역수',
    '환자수_인구보정', '위험노출시간_norm', '환자수_norm', '병원수_norm',
    '공원면적_norm', '금연구역수_norm', '건강위험도_점수', '건강위험도_등급', '계절'
]
df_all['행정동_코드'] = df_all['행정동_코드'].astype(int)
df_final = df_all[final_cols].copy()
df_final['건강위험도_점수'] = df_final['건강위험도_점수'].round().astype(int)

# 누락 동: 같은 구 평균으로 채우기
gu_avg = df_final.groupby(['자치구명', 'year', 'month']).agg(
    위험노출시간=('위험노출시간', 'mean'),
    pm25_평균=('pm25_평균', 'mean'), pm10_평균=('pm10_평균', 'mean'),
    온도_평균=('온도_평균', 'mean'), 습도_평균=('습도_평균', 'mean'),
    열지수_평균=('열지수_평균', 'mean'), 소음_평균=('소음_평균', 'mean'),
    high_temp_risk_비율=('high_temp_risk_비율', 'mean'),
    low_temp_risk_비율=('low_temp_risk_비율', 'mean'),
    humidity_risk_비율=('humidity_risk_비율', 'mean'),
    noise_risk_비율=('noise_risk_비율', 'mean'),
).reset_index()

missing_dongs = [
    {'행정동_코드': 11110690, '행정동_한글': '창신3동',    '자치구명': '종로구'},
    {'행정동_코드': 11170530, '행정동_한글': '남영동',     '자치구명': '용산구'},
    {'행정동_코드': 11350625, '행정동_한글': '중계2?3동',  '자치구명': '노원구'},
    {'행정동_코드': 11560515, '행정동_한글': '영등포본동', '자치구명': '영등포구'},
    {'행정동_코드': 11710631, '행정동_한글': '가락1동',    '자치구명': '송파구'},
]

aux_gu = df_final.drop_duplicates(subset=['자치구명'])[
    ['자치구명', '병원수', '공원면적', '금연구역수', '환자수_인구보정']
].reset_index(drop=True)

scalers = {}
for col in ['위험노출시간', '환자수_인구보정', '병원수', '공원면적', '금연구역수']:
    s = MinMaxScaler()
    s.fit(df_final[[col]])
    scalers[col] = s

fill_rows = []
for dong_info in missing_dongs:
    gu     = dong_info['자치구명']
    gu_data = gu_avg[gu_avg['자치구명'] == gu].copy()
    gu_aux  = aux_gu[aux_gu['자치구명'] == gu].iloc[0]
    for _, row in gu_data.iterrows():
        new_row = row.to_dict()
        new_row.update({
            '행정동_코드': dong_info['행정동_코드'],
            '행정동_한글': dong_info['행정동_한글'],
            '병원수': gu_aux['병원수'], '공원면적': gu_aux['공원면적'],
            '금연구역수': gu_aux['금연구역수'], '환자수_인구보정': gu_aux['환자수_인구보정'],
            '계절': '봄' if row['month'] in [3,4,5] else '여름' if row['month'] in [6,7,8] else '가을' if row['month'] in [9,10,11] else '겨울'
        })
        fill_rows.append(new_row)

df_fill = pd.DataFrame(fill_rows)
df_fill['위험노출시간_norm'] = scalers['위험노출시간'].transform(df_fill[['위험노출시간']])
df_fill['환자수_norm']       = scalers['환자수_인구보정'].transform(df_fill[['환자수_인구보정']])
df_fill['병원수_norm']       = 1 - scalers['병원수'].transform(df_fill[['병원수']])
df_fill['공원면적_norm']     = 1 - scalers['공원면적'].transform(df_fill[['공원면적']])
df_fill['금연구역수_norm']   = 1 - scalers['금연구역수'].transform(df_fill[['금연구역수']])
df_fill['건강위험도_점수'] = (
    df_fill['위험노출시간_norm'] * 0.73 + df_fill['병원수_norm'] * 0.10 +
    df_fill['환자수_norm'] * 0.07 + df_fill['공원면적_norm'] * 0.08 +
    df_fill['금연구역수_norm'] * 0.05 - 0.03
) * 100
df_fill['건강위험도_점수'] = df_fill['건강위험도_점수'].clip(0, 100).round().astype(int)
df_fill['건강위험도_등급'] = df_fill['건강위험도_점수'].apply(assign_grade)

# 합치기 → 중복 제거 → 상일동 제거
df_final = pd.concat([df_final, df_fill[df_final.columns]], ignore_index=True)
df_final = df_final.drop_duplicates(subset=['행정동_코드', 'year', 'month'], keep='first').reset_index(drop=True)
df_final = df_final[df_final['행정동_코드'] != 11740520].reset_index(drop=True)

# 최종 저장
df_final.to_csv(rf'{BASE_PATH}\monthly_risk.csv', index=False, encoding='utf-8-sig')

print(f'최종 저장 완료')
print(f'총 {len(df_final):,}행 | {df_final["행정동_코드"].nunique()}개 동')
print(f'연도: {sorted(df_final["year"].unique())}')
print(df_final['건강위험도_점수'].describe())

# EDA 시각화

In [ ]:
gu_score = df_final.groupby('자치구명')['건강위험도_점수'].mean().sort_values(ascending=False)
plt.figure(figsize=(12, 6))
gu_score.plot(kind='bar', color='salmon')
plt.title('서울 구별 평균 건강위험도 점수')
plt.xlabel('자치구'); plt.ylabel('평균 점수')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

### 10-2. 월별 트렌드

In [ ]:
monthly_trend = df_final.groupby('month')['건강위험도_점수'].mean()
plt.figure(figsize=(10, 5))
plt.plot(monthly_trend.index, monthly_trend.values, marker='o', color='steelblue', linewidth=2)
plt.title('서울 월별 평균 건강위험도 트렌드')
plt.xlabel('월'); plt.ylabel('평균 점수')
plt.xticks(range(1, 13)); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

### 10-3. 구별 월별 히트맵

In [ ]:
heatmap_score = df_final.groupby(['자치구명', 'month'])['건강위험도_점수'].mean().unstack()
plt.figure(figsize=(14, 10))
sns.heatmap(heatmap_score, annot=False, cmap='YlOrRd', linewidths=0.5)
plt.title('서울 구별 월별 건강위험도 히트맵')
plt.xlabel('월'); plt.ylabel('자치구'); plt.tight_layout(); plt.show()

### 10-4. Top/Bottom 10 행정동

In [ ]:
dong_score = df_final.groupby(['자치구명', '행정동_한글'])['건강위험도_점수'].mean().reset_index()
dong_score = dong_score.sort_values('건강위험도_점수', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
top10    = dong_score.head(10).copy()
bottom10 = dong_score.tail(10).copy()
top10['동네']    = top10['자치구명']    + ' ' + top10['행정동_한글']
bottom10['동네'] = bottom10['자치구명'] + ' ' + bottom10['행정동_한글']

axes[0].barh(top10['동네'], top10['건강위험도_점수'], color='crimson')
axes[0].set_title('건강위험도 Top 10 동네'); axes[0].invert_yaxis()
axes[1].barh(bottom10['동네'], bottom10['건강위험도_점수'], color='steelblue')
axes[1].set_title('건강위험도 Bottom 10 동네'); axes[1].invert_yaxis()
plt.tight_layout(); plt.show()

### 10-5. 자치구별 행정동 인터랙티브 조회

In [ ]:
gu_dropdown = widgets.Dropdown(
    options=sorted(df_final['자치구명'].unique()), value='강남구', description='자치구 선택:'
)

def plot_dong(gu):
    gu_data   = df_final[df_final['자치구명'] == gu]
    dong_sc   = gu_data.groupby('행정동_한글')['건강위험도_점수'].mean().sort_values(ascending=False)
    colors    = ['crimson' if s >= q80 else 'orange' if s >= q40 else 'steelblue' for s in dong_sc.values]
    plt.figure(figsize=(10, len(dong_sc) * 0.4 + 2))
    plt.barh(dong_sc.index, dong_sc.values, color=colors)
    plt.title(f'{gu} 행정동별 건강위험도 점수'); plt.xlabel('평균 점수')
    plt.gca().invert_yaxis(); plt.tight_layout(); plt.show()

widgets.interact(plot_dong, gu=gu_dropdown)

# 건강위험도 -> 건강안전도 등급 산정 방식 반전

In [10]:
import pandas as pd
import numpy as np

df = pd.read_csv(r'C:\Users\T\Desktop\SeoulMate\monthly_risk.csv', encoding='utf-8-sig')

# 건강위험도 → 건강안전도 (점수/등급 반전)
df['건강위험도_점수'] = 100 - df['건강위험도_점수']
df['건강위험도_등급'] = 6 - df['건강위험도_등급']

# 컬럼명 변경
df = df.rename(columns={
    '건강위험도_점수': '건강안전도_점수',
    '건강위험도_등급': '건강안전도_등급'
})

# 확인
print("점수 최고값 (안전한 동네):")
print(df.nlargest(3, '건강안전도_점수')[['행정동_한글', '건강안전도_점수', '건강안전도_등급']])
print("\n점수 최저값 (위험한 동네):")
print(df.nsmallest(3, '건강안전도_점수')[['행정동_한글', '건강안전도_점수', '건강안전도_등급']])
print("\n등급 분포:")
print(df['건강안전도_등급'].value_counts().sort_index())

# 저장
df.to_csv(r'C:\Users\T\Desktop\SeoulMate\monthly_risk.csv', index=False, encoding='utf-8-sig')
print("\n 저장 완료!")

KeyError: '건강위험도_점수'

In [ ]:
df['건강안전도_등급'] = 6 - df['건강안전도_등급']

print("점수 최고값 (안전한 동네):")
print(df.nlargest(3, '건강안전도_점수')[['행정동_한글', '건강안전도_점수', '건강안전도_등급']])
print("\n점수 최저값 (위험한 동네):")
print(df.nsmallest(3, '건강안전도_점수')[['행정동_한글', '건강안전도_점수', '건강안전도_등급']])
print("\n등급 분포:")
print(df['건강안전도_등급'].value_counts().sort_index())

df.to_csv(r'C:\Users\T\Desktop\SeoulMate\monthly_risk.csv', index=False, encoding='utf-8-sig')
print("\n저장 완료!")

점수 최고값 (안전한 동네):
     행정동_한글  건강안전도_점수  건강안전도_등급
1827   보라매동        95         1
1829   보라매동        95         1
1830   보라매동        95         1

점수 최저값 (위험한 동네):
     행정동_한글  건강안전도_점수  건강안전도_등급
5648   중곡3동         6         5
1051   성내3동        18         5
2216   중곡3동        18         5

등급 분포:
건강안전도_등급
1    1318
2    1324
3    1328
4    1344
5    1325
Name: count, dtype: int64

✅ 저장 완료!


In [14]:
safety = pd.read_csv(r'C:\Users\T\Desktop\SeoulMate\safety_model.csv', encoding='utf-8-sig')
print(safety.shape)
print(safety[['자치구', '2024_점수', '2024_등급']].to_string())

(25, 20)
     자치구  2024_점수  2024_등급
0    종로구     30.0        5
1     중구     30.0        5
2    용산구     30.7        5
3    성동구     73.3        1
4    광진구     44.8        4
5   동대문구     43.5        4
6    중랑구     50.8        3
7    성북구     66.2        1
8    강북구     47.0        4
9    도봉구     61.2        2
10   노원구     64.7        1
11   은평구     63.0        1
12  서대문구     76.6        1
13   마포구     41.8        4
14   양천구     55.8        3
15   강서구     30.6        5
16   구로구     56.6        3
17   금천구     61.3        2
18  영등포구     37.1        5
19   동작구     59.2        2
20   관악구     37.3        4
21   서초구     50.5        3
22   강남구     60.0        2
23   송파구     58.4        2
24   강동구     51.8        3


In [15]:
import pandas as pd
df = pd.read_csv(r'C:\Users\T\Desktop\SeoulMate\monthly_risk.csv', encoding='utf-8-sig')

# 이상한 케이스 확인
check = ['양평2동', '등촌1동', '신정1동']
print(df[df['행정동_한글'].isin(check)][['행정동_코드', '행정동_한글', '자치구명']].drop_duplicates())

        행정동_코드 행정동_한글  자치구명
836   11560620   양평2동   강북구
916   11500520   등촌1동   강동구
2428  11470620   신정1동   종로구
4684  11560620   양평2동  영등포구


In [16]:
import pandas as pd

df = pd.read_csv(r'C:\Users\T\Desktop\SeoulMate\monthly_risk.csv', encoding='utf-8-sig')
dong_code = pd.read_csv(r'C:\Users\T\Desktop\SeoulMate\dong_code.csv', encoding='cp949')

# dong_code.csv에 자치구 정보 있는지 확인
print(dong_code.columns.tolist())
print(dong_code.head(3))

['행정동_코드', '행정동_명', '엑스좌표_값', '와이좌표_값', '영역_면적']
     행정동_코드  행정동_명  엑스좌표_값  와이좌표_값    영역_면적
0  11110515  청운효자동  197342  453874  2568432
1  11110530    사직동  197383  452705  1158536
2  11110540    삼청동  198340  454312  1479255


In [18]:
import pandas as pd

df = pd.read_csv(r'C:\Users\T\Desktop\SeoulMate\monthly_risk.csv', encoding='utf-8-sig')
dong_eng = pd.read_csv(r'C:\Users\T\Desktop\SeoulMate\dong_eng.csv', encoding='cp949')

# 읍면동명 → 시군구명 매핑
dong_to_gu = dict(zip(dong_eng['읍면동명'], dong_eng['시군구명']))

# 행정동_한글 기준으로 자치구명 교정
df['자치구명_교정'] = df['행정동_한글'].map(dong_to_gu)

# 교정 전후 비교
diff = df[df['자치구명'] != df['자치구명_교정']][['행정동_코드', '행정동_한글', '자치구명', '자치구명_교정']].drop_duplicates()
print(f"불일치 동: {len(diff)}개")
print(diff.to_string())

불일치 동: 53개
          행정동_코드       행정동_한글  자치구명  자치구명_교정
755     11305534          삼양동   강북구      제주시
836     11560620         양평2동   강북구     영등포구
916     11500520         등촌1동   강동구      강서구
928     11500530         등촌2동   강동구      강서구
1025    11740640         성내1동   강동구       중구
1037    11740650         성내2동   강동구       중구
1049    11740660         성내3동   강동구       중구
1061    11110700         숭인1동   강동구      종로구
1173    11680740         일원2동   강남구      NaN
1197    11680521         논현1동   강남구      남동구
1209    11680531         논현2동   강남구      남동구
1382    11500603         가양1동   강서구       동구
1394    11500604         가양2동   강서구       동구
1834    11620545          청림동   관악구   포항시 남구
1844    11620595          청룡동   관악구  천안시 동남구
1916    11620615          중앙동   관악구     서귀포시
1928    11620765          미성동   관악구      군산시
1952    11620630          남현동   관악구      제천시
1988    11110540          삼청동   관악구      종로구
1990    11620745          삼성동   관악구      양산시
2062    11110700         숭인1동   관악구      종로구

In [19]:
import pandas as pd

df = pd.read_csv(r'C:\Users\T\Desktop\SeoulMate\monthly_risk.csv', encoding='utf-8-sig')
dong_eng = pd.read_csv(r'C:\Users\T\Desktop\SeoulMate\dong_eng.csv', encoding='cp949')

# 행정동 코드 앞 5자리 → 구코드
# dong_eng에서 서울만 필터 후 구코드 추출
seoul_dong = dong_eng[dong_eng['시도명'] == '서울특별시'].copy()

# 읍면동코드가 없으므로 행정동_코드 기준으로 매핑 딕셔너리 만들기
# dong_code와 dong_eng 연결: 읍면동명 + 시군구명으로 코드 매핑
dong_code = pd.read_csv(r'C:\Users\T\Desktop\SeoulMate\dong_code.csv', encoding='cp949')

# dong_code에 구 정보 붙이기 (코드 앞 5자리 기준)
# 서울 구코드 매핑
gu_code_map = {
    '11110': '종로구', '11140': '중구',   '11170': '용산구',
    '11200': '성동구', '11215': '광진구', '11230': '동대문구',
    '11260': '중랑구', '11290': '성북구', '11305': '강북구',
    '11320': '도봉구', '11350': '노원구', '11380': '은평구',
    '11410': '서대문구', '11440': '마포구', '11470': '양천구',
    '11500': '강서구', '11530': '구로구', '11545': '금천구',
    '11560': '영등포구', '11590': '동작구', '11620': '관악구',
    '11650': '서초구', '11680': '강남구', '11710': '송파구',
    '11740': '강동구'
}

# 행정동_코드 앞 5자리로 자치구명 교정
df['행정동_코드_str'] = df['행정동_코드'].astype(str).str[:10]
df['자치구명_교정'] = df['행정동_코드'].astype(str).str[:5].map(gu_code_map)

# 상일제1동, 상일제2동 특수처리
df.loc[df['행정동_코드'] == 1174052500, '자치구명_교정'] = '강동구'
df.loc[df['행정동_코드'] == 1174052600, '자치구명_교정'] = '강동구'

# 교정 적용
df['자치구명'] = df['자치구명_교정']
df = df.drop(columns=['행정동_코드_str', '자치구명_교정'])

# 확인
print(f"결측치: {df['자치구명'].isnull().sum()}")
check = df[df['행정동_한글'].isin(['양평2동', '등촌1동', '신정1동'])][['행정동_코드', '행정동_한글', '자치구명']].drop_duplicates()
print(check.to_string())

# 저장
df.to_csv(r'C:\Users\T\Desktop\SeoulMate\monthly_risk.csv', index=False, encoding='utf-8-sig')
print("\n✅ 저장 완료!")

결측치: 0
        행정동_코드 행정동_한글  자치구명
836   11560620   양평2동  영등포구
916   11500520   등촌1동   강서구
2428  11470620   신정1동   양천구

✅ 저장 완료!


In [22]:
import pandas as pd

df = pd.read_csv(r'C:\Users\T\Desktop\SeoulMate\monthly_risk.csv', encoding='utf-8-sig')

# 상일제1동/2동 코드 8자리로 수정
df.loc[df['행정동_코드'] == 1174052500, '행정동_코드'] = 11740525
df.loc[df['행정동_코드'] == 1174052600, '행정동_코드'] = 11740526

# 확인
print(df[df['행정동_한글'].isin(['상일제1동', '상일제2동'])][['행정동_코드', '행정동_한글', '자치구명']].drop_duplicates())

df.to_csv(r'C:\Users\T\Desktop\SeoulMate\monthly_risk.csv', index=False, encoding='utf-8-sig')
print("✅ 저장 완료!")

        행정동_코드 행정동_한글 자치구명
4886  11740525  상일제1동  강동구
4898  11740526  상일제2동  강동구
✅ 저장 완료!


In [23]:
import asyncio
from motor.motor_asyncio import AsyncIOMotorClient

async def clean():
    client = AsyncIOMotorClient('mongodb://localhost:27017')
    db = client['seoulmate']
    result = await db['health_scores'].delete_many({
        'dong_code': {'$in': [1174052500, 1174052600]}
    })
    print(f'삭제 완료: {result.deleted_count}건')
    client.close()

await clean()

삭제 완료: 28건


In [24]:
import pandas as pd

df = pd.read_csv(r'C:\Users\T\Desktop\SeoulMate\monthly_risk.csv', encoding='utf-8-sig')

# 10자리로 변환 (뒤에 00 붙이기)
df['행정동_코드'] = df['행정동_코드'].astype(str) + '00'
df['행정동_코드'] = df['행정동_코드'].astype(int)

# 확인
print(df['행정동_코드'].head(5))
print(f"코드 길이 확인: {df['행정동_코드'].astype(str).str.len().unique()}")

df.to_csv(r'C:\Users\T\Desktop\SeoulMate\monthly_risk.csv', index=False, encoding='utf-8-sig')
print("✅ 저장 완료!")

0    1132069000
1    1132069000
2    1132069000
3    1132069000
4    1132069000
Name: 행정동_코드, dtype: int64
코드 길이 확인: [10]
✅ 저장 완료!


In [27]:
import asyncio
from motor.motor_asyncio import AsyncIOMotorClient

async def clean():
    client = AsyncIOMotorClient('mongodb://localhost:27017')
    db = client['seoulmate']
    
    for col in ['health_scores', 'comfort_scores', 'safety_scores']:
        result = await db[col].delete_many({})
        print(f'{col} 삭제: {result.deleted_count}건')
    
    client.close()

await clean()

health_scores 삭제: 13278건
comfort_scores 삭제: 6768건
safety_scores 삭제: 13278건


In [2]:
print(df.columns.tolist())

['행정동_코드', '행정동_한글', '자치구명', 'year', 'month', '위험노출시간', 'pm25_평균', 'pm10_평균', '온도_평균', '습도_평균', '열지수_평균', '소음_평균', 'high_temp_risk_비율', 'low_temp_risk_비율', 'humidity_risk_비율', 'noise_risk_비율', '병원수', '공원면적', '금연구역수', '환자수_인구보정', '위험노출시간_norm', '환자수_norm', '병원수_norm', '공원면적_norm', '금연구역수_norm', '건강안전도_점수', '건강안전도_등급', '계절']


In [5]:
df['행정동_한글'] = df['행정동_한글'].str.replace('상일제1동', '상일1동', regex=False)
df['행정동_한글'] = df['행정동_한글'].str.replace('상일제2동', '상일2동', regex=False)

# 확인
print(df[df['행정동_한글'].str.contains('상일', na=False)]['행정동_한글'].unique())

# 저장
df.to_csv(r'C:\Users\T\Desktop\SeoulMate\monthly_risk.csv', index=False)
print("저장 완료!")

['상일1동' '상일2동']
저장 완료!


In [7]:
import re

def extract_dong(address):
    if pd.isna(address):
        return None
    # 구 다음에 오는 동만 추출
    match = re.search(r'[가-힣]+구\s+(\S+동\d*가?)', address)
    return match.group(1) if match else None

h_open['동'] = h_open['지번주소'].apply(extract_dong)

print(h_open[['지번주소', '동']].head(20).to_string())
print()
print('동 추출 실패 건수:', h_open['동'].isna().sum())
print()
# 실패한 주소 확인
print('=== 추출 실패 주소 샘플 ===')
print(h_open[h_open['동'].isna()]['지번주소'].dropna().head(10).to_string())

                              지번주소      동
3           서울특별시 성동구 용답동 2-3 장안빌딩    용답동
4         서울특별시 은평구 대조동 10-1 3~11층    대조동
5                              NaN   None
6             서울특별시 강서구 등촌동 645-11    등촌동
9         서울특별시 강남구 청담동 47-4 우리들병원    청담동
11   서울특별시 강남구 삼성동 158-12 서영빌딩 14층    삼성동
14                             NaN   None
20       서울특별시 강남구 대치동 889-11 대치빌딩    대치동
22              서울특별시 강남구 논현동 62-9    논현동
23                             NaN   None
27       서울특별시 성동구 성수동2가 279번지 44호  성수동2가
28          서울특별시 성동구 하왕십리동 1070-3  하왕십리동
29         서울특별시 성동구 용답동 99-2 남주빌딩    용답동
30           서울특별시 광진구 중곡동 30번지 1호    중곡동
31  서울특별시 광진구 화양동 4번지 12호 , 4번지19호    화양동
32          서울특별시 광진구 자양동 627번지 3호    자양동
35                             NaN   None
37         서울특별시 광진구 구의2동 80번지 25호   구의2동
39                             NaN   None
40        서울특별시 금천구 독산동 899-6 2~9층    독산동

동 추출 실패 건수: 220

=== 추출 실패 주소 샘플 ===
92                 서울특별시 동대문구 경희대길 45(회기동 산1번지)
139            서울특별시 동대문구 하정로 22

In [5]:
h_open = h[h['영업상태명'] == '영업/정상']
print(f'영업중 병원 수: {len(h_open)}')
print()
print(h_open['지번주소'].dropna().head(20).to_string())

영업중 병원 수: 555

3             서울특별시 성동구 용답동 2-3 장안빌딩
4           서울특별시 은평구 대조동 10-1 3~11층
6               서울특별시 강서구 등촌동 645-11
9           서울특별시 강남구 청담동 47-4 우리들병원
11     서울특별시 강남구 삼성동 158-12 서영빌딩 14층
20         서울특별시 강남구 대치동 889-11 대치빌딩
22                서울특별시 강남구 논현동 62-9
27         서울특별시 성동구 성수동2가 279번지 44호
28            서울특별시 성동구 하왕십리동 1070-3
29           서울특별시 성동구 용답동 99-2 남주빌딩
30             서울특별시 광진구 중곡동 30번지 1호
31    서울특별시 광진구 화양동 4번지 12호 , 4번지19호
32            서울특별시 광진구 자양동 627번지 3호
37           서울특별시 광진구 구의2동 80번지 25호
40          서울특별시 금천구 독산동 899-6 2~9층
45            서울특별시 은평구 구산동 191번지 1호
49           서울특별시 은평구 대조동 15번지 123호
52           서울특별시 은평구 녹번동 154번지 19호
53           서울특별시 은평구 대조동 187번지 51호
54           서울특별시 은평구 갈현동 462번지 52호


In [5]:
import pandas as pd

df = pd.read_csv(r'C:\Users\T\Desktop\SeoulMate\monthly_risk.csv')

# 상일1동 2026년 데이터 복사
sang1_2026 = df[(df['행정동_코드'] == 1174052500) & (df['year'] == 2026)].copy()

# 상일2동으로 코드랑 이름 변경
sang1_2026['행정동_코드'] = 1174052600
sang1_2026['행정동_한글'] = '상일2동'

# 기존 df에 추가
df = pd.concat([df, sang1_2026], ignore_index=True)

# 확인
print(df[df['행정동_코드'] == 1174052600][['행정동_코드', '행정동_한글', 'year', 'month']].tail(10))

# 저장
df.to_csv(r'C:\Users\T\Desktop\SeoulMate\monthly_risk.csv', index=False)
print('저장 완료!')

          행정동_코드 행정동_한글  year  month
4904  1174052600   상일2동  2025      7
4905  1174052600   상일2동  2025      8
4906  1174052600   상일2동  2025      9
4907  1174052600   상일2동  2025     10
4908  1174052600   상일2동  2025     11
4909  1174052600   상일2동  2025     12
6639  1174052600   상일2동  2026      1
6640  1174052600   상일2동  2026      2
6641  1174052600   상일2동  2026      3
6642  1174052600   상일2동  2026      4
저장 완료!


In [4]:
import pandas as pd
df = pd.read_csv(r'C:\Users\T\Desktop\SeoulMate\server\data\comfort_model.csv', encoding='utf-8-sig')
print(df.columns.tolist())
print(df.head(2))

['행정동_ID', '행정동코드', '년도', '월', '자치구', '행정동', '온도 평균', '습도 평균', '자외선 평균', '소음 평균', '미세먼지', '초미세먼지', '쾌적도점수', '쾌적도등급']
     행정동_ID       행정동코드    년도  월  자치구  행정동  온도 평균  습도 평균  자외선 평균  소음 평균  미세먼지  \
0  11010530  1111053000  2025  1  종로구  사직동    0.7   54.6     0.2   48.0  43.2   
1  11010540  1111054000  2025  1  종로구  삼청동    0.7   54.6     0.5   47.3  43.2   

   초미세먼지  쾌적도점수  쾌적도등급  
0   31.4   23.7      5  
1   31.4   23.4      5  
